# Evaluate a pretrained baseline from HuggingFace

Loads a finished model from the Hub, runs prediction over the test splits, and writes
prediction + metric files into `models/` in Drive using the **same** `MetricGenerator`
the AL runs use -- so the CSVs are directly comparable, column for column.

No training happens here.

**Edit the one cell below, then `Runtime -> Run all`.**

## Configuration -- the only cell you need to edit

In [ ]:
REPO_DIR = '/content/drive/MyDrive/AL-NLI/nli-training-example'

# Baseline to evaluate. Any Hub id with a 4-way sequence-classification head works.
#   'Flaglab/ESNLIR-RoBERTa'
#   'Flaglab/ESNLIR-XLM-RoBERTa'
MODEL_ID = 'Flaglab/ESNLIR-RoBERTa'

# These checkpoints ship only config.json + model.safetensors -- NO tokenizer files --
# so the tokenizer must come from the base model. None = infer from (model_type,
# vocab_size) in the checkpoint's config; set explicitly to override.
TOKENIZER_ID = None

# Where the data lives. Staged from Drive to local disk if absolute (see the AL notebook).
DATA_DIR = '/content/data'
SOURCE_DATA_SUBDIR = 'data'

# Splits to evaluate. None = every test*.json in DATA_DIR. Add 'val' to include validation.
EVAL_SPLITS = None
INCLUDE_VAL = True    # the AL curve is on validation, so the baseline needs it too

MAX_LEN    = 256      # must match how the baseline was trained
BATCH_SIZE = 256      # inference only
ONLY_PREMISE = False

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys

assert os.path.isdir(REPO_DIR), f'{REPO_DIR} not found in Drive'
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
OUT_DIR = os.path.join('models', 'baseline_' + MODEL_ID.split('/')[-1])
print('repo   :', os.getcwd())
print('model  :', MODEL_ID)
print('outputs:', OUT_DIR)

In [ ]:
!pip install -q "transformers>=4.53,<5" "accelerate>=1.8" "scikit-learn>=1.7"

import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (slow)')

## Data

Staged from Drive to local disk, same as the AL notebook.

In [ ]:
import os, shutil, time

FILES = ['train.json', 'val.json', 'test.json', 'test_full.jsonl']

if os.path.isabs(DATA_DIR):
    src_dir = os.path.join(REPO_DIR, SOURCE_DATA_SUBDIR)
    assert os.path.isdir(src_dir), f'{src_dir} not found in Drive'
    os.makedirs(DATA_DIR, exist_ok=True)
    for fname in FILES:
        if fname == 'train.json':
            continue                     # never needed for evaluation
        src, dst = os.path.join(src_dir, fname), os.path.join(DATA_DIR, fname)
        if not os.path.exists(src):
            print(f'  {fname}: not in Drive, skipped'); continue
        if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
            print(f'  {fname}: already staged'); continue
        t0 = time.time(); shutil.copyfile(src, dst)
        mb = os.path.getsize(dst)/1e6
        print(f'  {fname}: {mb:,.0f} MB in {time.time()-t0:.0f}s')
else:
    assert os.path.isdir(DATA_DIR), f'{DATA_DIR} not found'

# Which splits to score
candidates = [f for f in sorted(os.listdir(DATA_DIR))
              if 'test' in f and '.json' in f]
if INCLUDE_VAL and os.path.exists(os.path.join(DATA_DIR, 'val.json')):
    candidates.append('val.json')
splits = {os.path.splitext(f)[0]: os.path.join(DATA_DIR, f) for f in candidates}
if EVAL_SPLITS is not None:
    missing = set(EVAL_SPLITS) - set(splits)
    assert not missing, f'requested {sorted(missing)}; available: {sorted(splits)}'
    splits = {k: v for k, v in splits.items() if k in EVAL_SPLITS}
print('\nsplits to evaluate:', list(splits))

In [ ]:
import io, zipfile, urllib.request

# metrics.zip in these repos is this project's own MetricGenerator output. Its CSV headers
# carry the class order the checkpoint was trained with -- the only way to recover the
# mapping, since config.json exposes just LABEL_0..LABEL_3.
ZIP_CLASSES = None
try:
    raw = urllib.request.urlopen(
        f'https://huggingface.co/{MODEL_ID}/resolve/main/metrics.zip', timeout=60).read()
    zf = zipfile.ZipFile(io.BytesIO(raw))
    names = zf.namelist()
    print(f'metrics.zip: {len(names)} files, e.g. {names[:6]}')
    hit = next((n for n in names if n.endswith(('class_accuracy.csv', 'confusion_matrix.csv'))), None)
    if hit:
        header = zf.open(hit).readline().decode().strip().split(',')
        ZIP_CLASSES = [h.strip().lower() for h in header[1:]
                       if h.strip().lower() not in ('data_split', 'mean_accuracy')]
        print(f'\nclass order recovered from {hit}:\n  {ZIP_CLASSES}')
    else:
        print('no class_accuracy/confusion_matrix inside -- label order stays unverified')
except Exception as exc:
    print('could not read metrics.zip:', exc)

## Recover the label order, then load the model

`BERTDataset` builds targets with `pd.get_dummies`, i.e. **alphabetical** class order.
These checkpoints expose only `LABEL_0..LABEL_3`, so the order is taken from `metrics.zip`
when it can be read and assumed alphabetical otherwise. A wrong mapping looks exactly like
a model that predicts a single class, so the confusion matrix is worth checking.

In [ ]:
import torch
from transformers import AutoConfig, AutoModelForSequenceClassification
from esnlir.dataset_utils.dataset import BERTDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
hf_cfg = AutoConfig.from_pretrained(MODEL_ID)

KNOWN_TOKENIZERS = {
    ('roberta', 50262):      'bertin-project/bertin-roberta-base-spanish',
    ('xlm-roberta', 250002): 'FacebookAI/xlm-roberta-base',
}
TOK_ID = TOKENIZER_ID or KNOWN_TOKENIZERS.get((hf_cfg.model_type, hf_cfg.vocab_size))
assert TOK_ID, (
    f'No tokenizer known for model_type={hf_cfg.model_type}, vocab_size={hf_cfg.vocab_size}.\n'
    f'Set TOKENIZER_ID in the config cell to the base model this was fine-tuned from.'
)
print(f'model     : {MODEL_ID}  ({hf_cfg.model_type}, vocab {hf_cfg.vocab_size})')
print(f'tokenizer : {TOK_ID}')

probe_split, probe_path = next(iter(splits.items()))
probe_ds = BERTDataset(probe_path, MAX_LEN, TOK_ID, ONLY_PREMISE, None)
DATA_CLASSES = list(probe_ds.classes)
print('\ndataset classes (alphabetical):', DATA_CLASSES)

assert hf_cfg.num_labels == len(DATA_CLASSES), \
    f'{MODEL_ID} has {hf_cfg.num_labels} labels, data has {len(DATA_CLASSES)}'

model_labels = [str(hf_cfg.id2label[i]).strip().lower() for i in range(hf_cfg.num_labels)]
if ZIP_CLASSES and set(ZIP_CLASSES) == set(DATA_CLASSES):
    source, model_labels = 'metrics.zip', ZIP_CLASSES
elif all(l.startswith('label_') for l in model_labels):
    source, model_labels = 'ASSUMED alphabetical', list(DATA_CLASSES)
else:
    source = 'config.id2label'
print(f'checkpoint label order ({source}): {model_labels}')

if set(model_labels) != set(DATA_CLASSES):
    raise SystemExit(f'Cannot reconcile: model {model_labels} vs data {DATA_CLASSES}')
PERM = [model_labels.index(c) for c in DATA_CLASSES]
print(f'logit column permutation: {PERM}' + ('  (identity)' if PERM == sorted(PERM) else ''))
if source.startswith('ASSUMED'):
    print('\nWARNING: the order could not be verified. If accuracy lands near 0.25 and the\n'
          'confusion matrix is not roughly diagonal, the assumed order is wrong.')

model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(device).eval()
print('\nloaded on', device)

## Predict and write metrics

In [ ]:
import os, torch, numpy as np, pandas as pd
from torch.utils.data import DataLoader
from esnlir.evaluation.metric_generation import MetricGenerator

@torch.no_grad()
def predict(dataset):
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=2)
    logits, golds = [], []
    for batch in loader:
        # RoBERTa-family models ignore token_type_ids (type_vocab_size 1); passing only
        # ids+mask keeps this portable across checkpoints.
        out = model(input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device))
        logits.append(out.logits.float().cpu())
        golds.append(batch['labels'])
    return torch.cat(logits).numpy()[:, PERM], torch.cat(golds).numpy()

for split, path in splits.items():
    ds = probe_ds if split == probe_split else BERTDataset(path, MAX_LEN, TOK_ID, ONLY_PREMISE, None)
    assert list(ds.classes) == DATA_CLASSES, f'{split} has classes {ds.classes}'

    lg, gold = predict(ds)
    y_pred, y_true = lg.argmax(1), gold.argmax(1)

    metrics = MetricGenerator(
        y_true, y_pred, ds.datasets, ds.genres, ds.domains, split, DATA_CLASSES
    ).run()

    # Raw per-class probabilities, so any metric can be recomputed later without re-running
    probs = torch.softmax(torch.tensor(lg), dim=1).numpy()
    preds_df = pd.DataFrame({
        'true_label': [DATA_CLASSES[i] for i in y_true],
        'predicted_label': [DATA_CLASSES[i] for i in y_pred],
    })
    for j, name in enumerate(DATA_CLASSES):
        preds_df[f'p_{name}'] = probs[:, j]
    metrics[f'{split}/total/predictions'] = preds_df

    for rel, df in metrics.items():
        dest = os.path.join(OUT_DIR, f'{rel}.csv')
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        df.to_csv(dest)

    acc = (y_pred == y_true).mean()
    print(f'{split:<12} n={len(y_true):>7,}  accuracy={acc:.4f}  -> {OUT_DIR}/{split}/')

## Results

In [ ]:
import glob, os
import pandas as pd

for rep in sorted(glob.glob(os.path.join(OUT_DIR, '*', 'total', 'classification_report.csv'))):
    print('===', rep.split(os.sep)[-3], '===')
    display(pd.read_csv(rep, index_col=0))

for cm in sorted(glob.glob(os.path.join(OUT_DIR, '*', 'total', 'confusion_matrix.csv'))):
    print('=== confusion matrix:', cm.split(os.sep)[-3], '(rows = true) ===')
    display(pd.read_csv(cm, index_col=0))

In [ ]:
# Side-by-side with an AL run, if one is present in Drive
import glob, os
import pandas as pd

rows = []
for path in glob.glob('models/*/**/total/general_stats.csv', recursive=True):
    parts = path.split(os.sep)
    split = parts[-3]
    run = parts[1]
    stage = parts[-4] if len(parts) > 4 else ''
    df = pd.read_csv(path, index_col=0)
    rows.append({'run': run, 'stage': stage, 'split': split,
                 'accuracy': df['accuracy'].iloc[0], 'f1_score': df['f1_score'].iloc[0]})

if rows:
    comp = pd.DataFrame(rows).sort_values(['split', 'f1_score'], ascending=[True, False])
    display(comp.reset_index(drop=True))
else:
    print('no general_stats.csv found under models/')

## Notes

* Output layout matches the AL runs exactly (`<split>/total/*.csv` plus the `dataset/`,
  `genre/` and `domain/` breakdowns), so the two are directly comparable. One addition:
  `<split>/total/predictions.csv` carries per-class probabilities, letting you recompute
  metrics or build ensembles without re-running inference.
* **Compare on the same split.** `test_full` is 1,695 rows and imbalanced (37% `neutral`),
  so accuracy on it runs high relative to macro-F1. `test` is 80,216 rows and perfectly
  balanced -- the stronger number to quote.
* **`MAX_LEN` must match how the baseline was trained.** It is not read from the
  checkpoint. If a baseline used a different length, its numbers are not comparable to
  the AL runs at 256.
* If the checkpoint exposes generic `LABEL_n` names the mapping cannot be verified. A
  wrong order shows up as one class with high recall and the others near zero -- check
  the confusion matrix before trusting the numbers.